In [1]:
!pip install -q transformers accelerate

In [2]:
import pandas as pd
import torch
import numpy as np
from torch.utils.data import DataLoader, Dataset
import re
import random
from collections import Counter, defaultdict
from transformers import (
    BertConfig,
    BertForMaskedLM,
    Trainer,
    TrainingArguments,
    set_seed
)

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
df = pd.read_csv("/content/transactions.tgz",compression='gzip', nrows=8_000_000)
df

,card_transaction.v1.csv,Card,Year,Month,Day,Time,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?
0,0,0,2002,9,1,06:21,$134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,NaN,No
1,0,0,2002,9,1,06:42,$38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
2,0,0,2002,9,2,06:22,$120.34,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
3,0,0,2002,9,2,17:45,$128.95,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.0,5651,NaN,No
4,0,0,2002,9,3,06:23,$104.71,Swipe Transaction,5817218446178736267,La Verne,CA,91750.0,5912,NaN,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7999995,680,2,2003,10,1,07:30,$61.82,Swipe Transaction,-7566024815690246185,Miami,FL,33179.0,8021,NaN,No
7999996,680,2,2003,10,2,06:58,$2.81,Swipe Transaction,-2744911404133435018,Miami,FL,33183.0,5812,NaN,No
7999997,680,2,2003,10,2,13:14,$7.20,Swipe Transaction,4722913068560264812,Miami,FL,33179.0,5411,NaN,No
7999998,680,2,2003,10,2,13:39,$53.76,Swipe Transaction,1799189980464955940,Miami,FL,33179.0,5499,NaN,No


In [5]:
df.isna().sum()

,0
card_transaction.v1.csv,0
Card,0
Year,0
Month,0
Day,0
Time,0
Amount,0
Use Chip,0
Merchant Name,0
Merchant City,0


## According to the dataset `card_transaction.v1.csv` is meant to be user

In [6]:
COLUMN_MAP = {
    "card_transaction.v1.csv": "user",
    "Card": "card",
    "Year": "year",
    "Month": "month",
    "Day": "day",
    "Time": "time",
    "Amount": "amount",
    "Use Chip": "use_chip",
    "Merchant Name": "merchant_name",
    "Merchant City": "merchant_city",
    "Merchant State": "merchant_state",
    "Zip": "zip",
    "MCC": "mcc",
    "Errors?": "errors",
    "Is Fraud?": "is_fraud",
}

df = df.rename(columns=COLUMN_MAP)
df

,user,card,year,month,day,time,amount,use_chip,merchant_name,merchant_city,merchant_state,zip,mcc,errors,is_fraud
0,0,0,2002,9,1,06:21,$134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,NaN,No
1,0,0,2002,9,1,06:42,$38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
2,0,0,2002,9,2,06:22,$120.34,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
3,0,0,2002,9,2,17:45,$128.95,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.0,5651,NaN,No
4,0,0,2002,9,3,06:23,$104.71,Swipe Transaction,5817218446178736267,La Verne,CA,91750.0,5912,NaN,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7999995,680,2,2003,10,1,07:30,$61.82,Swipe Transaction,-7566024815690246185,Miami,FL,33179.0,8021,NaN,No
7999996,680,2,2003,10,2,06:58,$2.81,Swipe Transaction,-2744911404133435018,Miami,FL,33183.0,5812,NaN,No
7999997,680,2,2003,10,2,13:14,$7.20,Swipe Transaction,4722913068560264812,Miami,FL,33179.0,5411,NaN,No
7999998,680,2,2003,10,2,13:39,$53.76,Swipe Transaction,1799189980464955940,Miami,FL,33179.0,5499,NaN,No


## Creating datetime

In [7]:
date = pd.to_datetime(
    dict(
        year=df["year"],
        month=df["month"],
        day=df["day"]
    )
)
time = pd.to_timedelta(
    df["time"].astype(str) + ":00"
)
df["timestamp"] = date + time
df

,user,card,year,month,day,time,amount,use_chip,merchant_name,merchant_city,merchant_state,zip,mcc,errors,is_fraud,timestamp
0,0,0,2002,9,1,06:21,$134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,NaN,No,2002-09-01 06:21:00
1,0,0,2002,9,1,06:42,$38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No,2002-09-01 06:42:00
2,0,0,2002,9,2,06:22,$120.34,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No,2002-09-02 06:22:00
3,0,0,2002,9,2,17:45,$128.95,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.0,5651,NaN,No,2002-09-02 17:45:00
4,0,0,2002,9,3,06:23,$104.71,Swipe Transaction,5817218446178736267,La Verne,CA,91750.0,5912,NaN,No,2002-09-03 06:23:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7999995,680,2,2003,10,1,07:30,$61.82,Swipe Transaction,-7566024815690246185,Miami,FL,33179.0,8021,NaN,No,2003-10-01 07:30:00
7999996,680,2,2003,10,2,06:58,$2.81,Swipe Transaction,-2744911404133435018,Miami,FL,33183.0,5812,NaN,No,2003-10-02 06:58:00
7999997,680,2,2003,10,2,13:14,$7.20,Swipe Transaction,4722913068560264812,Miami,FL,33179.0,5411,NaN,No,2003-10-02 13:14:00
7999998,680,2,2003,10,2,13:39,$53.76,Swipe Transaction,1799189980464955940,Miami,FL,33179.0,5499,NaN,No,2003-10-02 13:39:00


## Converting the amount to number

In [8]:
df["amount_numeric"] = df["amount"].astype(str).str.replace("$","").str.replace(",","")
df["amount_numeric"] = pd.to_numeric(df["amount_numeric"])
df.head()

,user,card,year,month,day,time,amount,use_chip,merchant_name,merchant_city,merchant_state,zip,mcc,errors,is_fraud,timestamp,amount_numeric
0,0,0,2002,9,1,06:21,$134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,NaN,No,2002-09-01 06:21:00,134.09
1,0,0,2002,9,1,06:42,$38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No,2002-09-01 06:42:00,38.48
2,0,0,2002,9,2,06:22,$120.34,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No,2002-09-02 06:22:00,120.34
3,0,0,2002,9,2,17:45,$128.95,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.0,5651,NaN,No,2002-09-02 17:45:00,128.95
4,0,0,2002,9,3,06:23,$104.71,Swipe Transaction,5817218446178736267,La Verne,CA,91750.0,5912,NaN,No,2002-09-03 06:23:00,104.71


In [9]:
df = df.sort_values(["user", "timestamp"]).reset_index(drop=True)

## Deriving some useful time features

In [10]:
df["hour"] = df["timestamp"].dt.hour

df["day_of_week"] = (
    df["timestamp"].dt.dayofweek
)

df["day_of_month"] = (
    df["timestamp"].dt.day
)

df["calendar_month"] = (
    df["timestamp"].dt.month
)

df["previous_time"] = (
    df.groupby("user")["timestamp"]
    .diff()
    .dt
    .total_seconds()
    .div(60)
)
df["previous_time"] = (
    df["previous_time"]
    .fillna(0)
    .clip(lower=0,upper=60*24*30)
)

df

,user,card,year,month,day,time,amount,use_chip,merchant_name,merchant_city,...,mcc,errors,is_fraud,timestamp,amount_numeric,hour,day_of_week,day_of_month,calendar_month,previous_time
0,0,0,2002,9,1,06:21,$134.09,Swipe Transaction,3527213246127876953,La Verne,...,5300,NaN,No,2002-09-01 06:21:00,134.09,6,6,1,9,0.0
1,0,0,2002,9,1,06:42,$38.48,Swipe Transaction,-727612092139916043,Monterey Park,...,5411,NaN,No,2002-09-01 06:42:00,38.48,6,6,1,9,21.0
2,0,0,2002,9,2,06:22,$120.34,Swipe Transaction,-727612092139916043,Monterey Park,...,5411,NaN,No,2002-09-02 06:22:00,120.34,6,0,2,9,1420.0
3,0,0,2002,9,2,17:45,$128.95,Swipe Transaction,3414527459579106770,Monterey Park,...,5651,NaN,No,2002-09-02 17:45:00,128.95,17,0,2,9,683.0
4,0,0,2002,9,3,06:23,$104.71,Swipe Transaction,5817218446178736267,La Verne,...,5912,NaN,No,2002-09-03 06:23:00,104.71,6,1,3,9,758.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7999995,680,1,2020,2,26,16:10,$26.98,Chip Transaction,272399770636553347,Miami,...,5411,NaN,No,2020-02-26 16:10:00,26.98,16,2,26,2,559.0
7999996,680,1,2020,2,27,12:41,$8.13,Chip Transaction,272399770636553347,Miami,...,5411,NaN,No,2020-02-27 12:41:00,8.13,12,3,27,2,1231.0
7999997,680,0,2020,2,28,06:49,$2.72,Chip Transaction,1799189980464955940,Miami,...,5499,NaN,No,2020-02-28 06:49:00,2.72,6,4,28,2,1088.0
7999998,680,1,2020,2,28,12:43,$6.94,Chip Transaction,272399770636553347,Miami,...,5411,NaN,No,2020-02-28 12:43:00,6.94,12,4,28,2,354.0


## Splitting the dataset

In [11]:
train_df = df[
    df["timestamp"] < pd.Timestamp("2017-01-01")
].copy()

val_df = df[
    (df["timestamp"] >= pd.Timestamp("2017-01-01"))
    & (df["timestamp"] < pd.Timestamp("2019-01-01"))
].copy()

test_df = df[
    df["timestamp"] >= pd.Timestamp("2019-01-01")
].copy()

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 6198642
Validation: 1123895
Test: 677463


In [12]:
import gc
del df
gc.collect()

272

In [13]:
dataset = [
    ("train", train_df),
    ("validation", val_df),
    ("test", test_df),
]

for name, part in dataset:
    print(
        name, part["is_fraud"].value_counts(
            normalize=True
        )
    )

train is_fraud
No     0.998717
Yes    0.001283
Name: proportion, dtype: float64
validation is_fraud
No     0.999197
Yes    0.000803
Name: proportion, dtype: float64
test is_fraud
No     0.999074
Yes    0.000926
Name: proportion, dtype: float64


## Quantization of the different features
This is used to ensure the values are in different boundaeies and not a continuous value

In [14]:
def fit_quantile_boundaries(values, n_bins):
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]
    quantiles = np.linspace(0, 1, n_bins + 1)[1:-1]
    boundaries = np.quantile(values, quantiles)
    boundaries = np.unique(boundaries)
    return boundaries


def apply_quantization(values, boundaries):
    values = np.asarray(values, dtype=np.float64)
    ind = np.digitize(values, boundaries, right=False)
    return ind.astype(dtype=np.int32)


In [15]:
amount_boundaries = fit_quantile_boundaries(train_df["amount_numeric"], 32)
for name, part in dataset:
    part["amount_bin"] = apply_quantization(part["amount_numeric"], amount_boundaries)

    part["timestamp_seconds"] = part["timestamp"].astype(np.int64) // 10**9

In [16]:
timestamp_boundaries = fit_quantile_boundaries(train_df["timestamp_seconds"], 64)

for name, part in dataset:
    part["timestamp_bin"] = apply_quantization(part["timestamp_seconds"], timestamp_boundaries)

In [17]:
delta_boundaries = fit_quantile_boundaries(train_df["previous_time"], 32)

for name, part in dataset:
    part["delta_bin"] = apply_quantization(part["previous_time"], delta_boundaries)

To clean the data and also clean ZIP

In [18]:
def clean_data(values):
    if pd.isna(values):
        return "NONE"

    values = str(values).strip().upper()
    values = re.sub( r"\s+", "_", values)
    values = values.replace("=","_")
    return values

def clean_zip(values):
    if pd.isna(values):
        return "NONE"

    try:
        number  = float(values)
        if number.is_integer():
            return str(int(number))

    except(TypeError, ValueError):
        pass
    return clean_data(values)

In [19]:
def transaction_to_tokens(row):

    return [
        "[TXN]",

        f"CARD={clean_data(row.card)}",

        f"TIMESTAMP={row.timestamp_bin}",

        f"HOUR={row.hour}",

        f"DOW={row.day_of_week}",

        f"MONTH={row.calendar_month}",

        f"DOM={row.day_of_month}",

        f"DELTA={row.delta_bin}",

        f"AMOUNT={row.amount_bin}",

        f"CHANNEL={clean_data(row.use_chip)}",

        f"MERCHANT={clean_data(row.merchant_name)}",

        f"CITY={clean_data(row.merchant_city)}",

        f"STATE={clean_data(row.merchant_state)}",

        f"ZIP={clean_zip(row.zip)}",

        f"MCC={clean_data(row.mcc)}",

        f"ERROR={clean_data(row.errors)}",
    ]

In [20]:
sample_row = next(
    train_df.itertuples(index=False)
)

transaction_to_tokens(
    sample_row
)

['[TXN]',
 'CARD=0',
 'TIMESTAMP=2',
 'HOUR=6',
 'DOW=6',
 'MONTH=9',
 'DOM=1',
 'DELTA=0',
 'AMOUNT=29',
 'CHANNEL=SWIPE_TRANSACTION',
 'MERCHANT=3527213246127876953',
 'CITY=LA_VERNE',
 'STATE=CA',
 'ZIP=91750',
 'MCC=5300',
 'ERROR=NONE']

In [21]:
def build_sequence(
    dataframe,
    transaction_per_sequence=16,
    stride=8,
    max_sequence=None
):
    sequences = []
    dataframe = dataframe.sort_values(["user", "timestamp"])

    for user_id, user_df in dataframe.groupby("user", sort=False):
        transactions = [
            transaction_to_tokens(row)
            for row in user_df.itertuples(index=False)
        ]
        if len(transactions) < transaction_per_sequence:
            continue

        for start in range(0, len(transactions) - transaction_per_sequence + 1, stride):
            window = transactions[start: start + transaction_per_sequence]
            flattened = []
            for transaction in window:
                flattened.extend(transaction)

            sequences.append(flattened)

            if (
                max_sequence is not None
                and len(sequences) >= max_sequence
            ):
                return sequences
    return sequences


In [22]:
train_sequences = build_sequence(
    train_df,
    transaction_per_sequence=16,
    stride=8,
    max_sequence=200_000
)
del dataset
del train_df
gc.collect()

val_sequences = build_sequence(
    val_df,
    transaction_per_sequence=16,
    stride=8,
    max_sequence=80_000
)

del val_df
gc.collect()

print(
    len(train_sequences),
    len(val_sequences)
)

200000 80000


In [23]:
print(train_sequences[0][:64])

['[TXN]', 'CARD=0', 'TIMESTAMP=2', 'HOUR=6', 'DOW=6', 'MONTH=9', 'DOM=1', 'DELTA=0', 'AMOUNT=29', 'CHANNEL=SWIPE_TRANSACTION', 'MERCHANT=3527213246127876953', 'CITY=LA_VERNE', 'STATE=CA', 'ZIP=91750', 'MCC=5300', 'ERROR=NONE', '[TXN]', 'CARD=0', 'TIMESTAMP=2', 'HOUR=6', 'DOW=6', 'MONTH=9', 'DOM=1', 'DELTA=5', 'AMOUNT=17', 'CHANNEL=SWIPE_TRANSACTION', 'MERCHANT=-727612092139916043', 'CITY=MONTEREY_PARK', 'STATE=CA', 'ZIP=91754', 'MCC=5411', 'ERROR=NONE', '[TXN]', 'CARD=0', 'TIMESTAMP=2', 'HOUR=6', 'DOW=0', 'MONTH=9', 'DOM=2', 'DELTA=29', 'AMOUNT=29', 'CHANNEL=SWIPE_TRANSACTION', 'MERCHANT=-727612092139916043', 'CITY=MONTEREY_PARK', 'STATE=CA', 'ZIP=91754', 'MCC=5411', 'ERROR=NONE', '[TXN]', 'CARD=0', 'TIMESTAMP=2', 'HOUR=17', 'DOW=0', 'MONTH=9', 'DOM=2', 'DELTA=23', 'AMOUNT=29', 'CHANNEL=SWIPE_TRANSACTION', 'MERCHANT=3414527459579106770', 'CITY=MONTEREY_PARK', 'STATE=CA', 'ZIP=91754', 'MCC=5651', 'ERROR=NONE']


In [24]:
SPECIAL_TOKENS = [
    "[PAD]",
    "[UNK]",
    "[CLS]",
    "[SEP]",
    "[MASK]",
    "[TXN]"
]


def get_field(token):

    if token.startswith("["):
        return None

    if "=" not in token:
        return None

    return token.split("=",1)[0]

In [25]:
field_counters = defaultdict(
    Counter
)

for sequence in train_sequences:

    for token in sequence:

        field = get_field(token)

        if field is not None:
            field_counters[field][token] += 1

In [26]:
FIELD_LIMITS = {
    "MERCHANT": 20_000,
    "CITY": 5_000,
    "ZIP": 10_000
}

field_counters.keys()

dict_keys(['CARD', 'TIMESTAMP', 'HOUR', 'DOW', 'MONTH', 'DOM', 'DELTA', 'AMOUNT', 'CHANNEL', 'MERCHANT', 'CITY', 'STATE', 'ZIP', 'MCC', 'ERROR'])

In [27]:
vocab_tokens = SPECIAL_TOKENS.copy()

fields = sorted(
    field_counters.keys()
)

# Add field-specific UNK tokens
for field in fields:
    vocab_tokens.append(
        f"[UNK_{field}]"
    )


for field in fields:

    counter = field_counters[field]

    items = [
        (token, count)
        for token, count
        in counter.most_common()
        if count >= 5
    ]

    limit = FIELD_LIMITS.get(
        field
    )

    if limit is not None:
        items = items[:limit]

    for token, count in items:

        if token not in vocab_tokens:
            vocab_tokens.append(token)

In [28]:
token_to_id = {
    token: idx
    for idx, token
    in enumerate(vocab_tokens)
}

id_to_token = {
    idx: token
    for token, idx
    in token_to_id.items()
}

print(
    "Vocabulary size:",
    len(token_to_id)
)

Vocabulary size: 22223


In [29]:
def encode_token(token):

    if token in token_to_id:
        return token_to_id[token]

    field = get_field(token)

    if field is not None:

        unk_token = f"[UNK_{field}]"

        if unk_token in token_to_id:
            return token_to_id[
                unk_token
            ]

    return token_to_id["[UNK]"]

In [30]:
TOKENS_PER_TRANSACTION = len(
    transaction_to_tokens(sample_row)
)

MAX_LENGTH = (
    16
    * TOKENS_PER_TRANSACTION
    + 2
)

print("Tokens / transaction:",TOKENS_PER_TRANSACTION)

print("Max sequence length:", MAX_LENGTH)

Tokens / transaction: 16
Max sequence length: 258


In [31]:
class FinancialTransactionDataset(Dataset):

    def __init__(
        self,
        sequences,
        max_length
    ):
        self.sequences = sequences
        self.max_length = max_length

        self.pad_id = token_to_id["[PAD]"]
        self.cls_id = token_to_id["[CLS]"]
        self.sep_id = token_to_id["[SEP]"]

    def __len__(self):
        return len(
            self.sequences
        )

    def __getitem__(self, idx):

        sequence = self.sequences[
            idx
        ]

        ids = [
            encode_token(token)
            for token in sequence
        ]

        ids = (
            [self.cls_id]
            + ids[
                :self.max_length - 2
            ]
            + [self.sep_id]
        )

        attention_mask = (
            [1] * len(ids)
        )

        padding = (
            self.max_length
            - len(ids)
        )

        ids += (
            [self.pad_id]
            * padding
        )

        attention_mask += (
            [0]
            * padding
        )

        return {
            "input_ids": torch.tensor(
                ids,
                dtype=torch.long
            ),
            "attention_mask": torch.tensor(
                attention_mask,
                dtype=torch.long
            ),
        }

In [32]:
train_dataset = (
    FinancialTransactionDataset(
        train_sequences,
        MAX_LENGTH
    )
)

val_dataset = (
    FinancialTransactionDataset(
        val_sequences,
        MAX_LENGTH
    )
)

In [33]:
field_to_ids = defaultdict(
    list
)

id_to_field = {}

for idx, token in id_to_token.items():

    field = get_field(token)

    id_to_field[idx] = field

    if field is not None:
        field_to_ids[field].append(idx)

In [34]:
field_unknown_ids = set()

for field in fields:

    token = f"[UNK_{field}]"

    field_unknown_ids.add(
        token_to_id[token]
    )

for field in field_to_ids:

    field_to_ids[field] = [
        idx
        for idx in field_to_ids[field]
        if idx not in field_unknown_ids
    ]

In [35]:
class FinancialMLMCollator:

    def __init__(
        self,
        mlm_probability=0.15
    ):

        self.mlm_probability = (mlm_probability)

        self.mask_id = (token_to_id["[MASK]"])

        self.pad_id = (token_to_id["[PAD]"])

        self.cls_id = (token_to_id["[CLS]"])

        self.sep_id = (token_to_id["[SEP]"])

        self.txn_id = (token_to_id["[TXN]"])

        self.never_mask = {
            self.pad_id,
            self.cls_id,
            self.sep_id,
            self.txn_id,
            *field_unknown_ids
        }

    def __call__(self, examples):

        input_ids = torch.stack([
            x["input_ids"]
            for x in examples
        ])

        attention_mask = torch.stack([
            x["attention_mask"]
            for x in examples
        ])

        labels = input_ids.clone()

        probability_matrix = torch.full(
            labels.shape,
            self.mlm_probability
        )

        for token_id in self.never_mask:

            probability_matrix.masked_fill_(
                input_ids == token_id,
                0.0
            )

        masked_indices = torch.bernoulli(probability_matrix).bool()

        labels[
            ~masked_indices
        ] = -100

        replace_with_mask = (
            torch.rand(
                labels.shape
            ) < 0.8
        ) & masked_indices

        input_ids[
            replace_with_mask
        ] = self.mask_id


        remaining = (
            masked_indices
            & ~replace_with_mask
        )

        replace_random = (
            torch.rand(
                labels.shape
            ) < 0.5
        ) & remaining

        positions = (
            replace_random
            .nonzero(
                as_tuple=False
            )
        )

        for batch_idx, pos_idx in positions:

            original_id = int(
                labels[
                    batch_idx,
                    pos_idx
                ]
            )

            field = id_to_field.get(
                original_id
            )

            candidates = field_to_ids.get(
                field,
                []
            )

            if candidates:

                random_id = random.choice(
                    candidates
                )

                input_ids[
                    batch_idx,
                    pos_idx
                ] = random_id

        return {
            "input_ids":
                input_ids,

            "attention_mask":
                attention_mask,

            "labels":
                labels
        }

In [36]:
collator = FinancialMLMCollator(
    mlm_probability=0.15
)

In [37]:
config = BertConfig(

    vocab_size=len(
        token_to_id
    ),

    hidden_size=128,

    num_hidden_layers=4,

    num_attention_heads=4,

    intermediate_size=512,

    hidden_dropout_prob=0.1,

    attention_probs_dropout_prob=0.1,

    max_position_embeddings=(
        MAX_LENGTH + 8
    ),

    pad_token_id=token_to_id[
        "[PAD]"
    ]
)

In [38]:
model = BertForMaskedLM(
    config
)

model.to(device)

BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(22223, 128, padding_idx=0)
      (position_embeddings): Embedding(266, 128)
      (token_type_embeddings): Embedding(2, 128)
      (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-3): 4 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=128, out_features=128, bias=True)
              (key): Linear(in_features=128, out_features=128, bias=True)
              (value): Linear(in_features=128, out_features=128, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=128, out_features=128, bias=True)
              (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_aff

In [39]:
number_parameters = sum(
    parameter.numel()
    for parameter
    in model.parameters()
)

print(
    f"{number_parameters / 1e6:.2f}M parameters"
)

3.71M parameters


In [40]:
loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collator
)

batch = next(
    iter(loader)
)

for key, value in batch.items():
    print(
        key,
        value.shape
    )

input_ids torch.Size([4, 258])
attention_mask torch.Size([4, 258])
labels torch.Size([4, 258])


In [41]:
batch = {
    key: value.to(device)
    for key, value
    in batch.items()
}

In [42]:
with torch.no_grad():

    output = model(
        **batch
    )

print(
    "Initial loss:",
    output.loss.item()
)

Initial loss: 10.019933700561523


In [43]:
training_args = TrainingArguments(

    output_dir=(
        "/content/"
        "financial_bert_checkpoints"
    ),

    num_train_epochs=12,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    gradient_accumulation_steps=4,

    learning_rate=3e-4,

    weight_decay=0.01,

    warmup_steps=100,

    logging_steps=100,

    eval_strategy="steps",
    eval_steps=500,

    save_strategy="steps",
    save_steps=500,

    save_total_limit=2,

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",

    greater_is_better=False,

    fp16=torch.cuda.is_available(),

    report_to="none",

    dataloader_num_workers=2
)

In [44]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=val_dataset,

    data_collator=collator
)

In [45]:
trainer.train()

Step,Training Loss,Validation Loss
500,12.194869,3.049850
1000,7.490043,2.064931
1500,6.284596,1.841560
2000,5.401938,1.605845
2500,4.824224,1.486501
3000,4.576165,1.418739
3500,4.270668,1.354308
4000,4.068292,1.308541
4500,3.901721,1.273590
5000,3.750134,1.236213


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['cls.predictions.decoder.weight', 'cls.predictions.decoder.bias'].


TrainOutput(global_step=37500, training_loss=2.8037612727864585, metrics={'train_runtime': 12744.2075, 'train_samples_per_second': 188.321, 'train_steps_per_second': 2.943, 'total_flos': 3092290992000000.0, 'train_loss': 2.8037612727864585, 'epoch': 12.0})

In [46]:
results = trainer.evaluate()

results

Training Loss,Validation Loss,Step
1.784233,0.630945,37500


{'eval_loss': 0.6309447288513184}

In [47]:
def evaluate_masked_accuracy(
    model,
    dataset,
    batch_size=32
):

    torch.manual_seed(1234)
    random.seed(1234)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collator
    )

    model.eval()

    total_correct = 0
    total_count = 0

    total_loss = 0.0
    total_loss_tokens = 0

    field_correct = defaultdict(int)
    field_total = defaultdict(int)

    with torch.no_grad():

        for batch in loader:

            batch = {
                key: value.to(device)
                for key, value
                in batch.items()
            }

            outputs = model(
                **batch
            )

            labels = batch[
                "labels"
            ]

            predictions = (
                outputs.logits.argmax(
                    dim=-1
                )
            )

            masked = (
                labels != -100
            )

            masked_count = (
                masked.sum().item()
            )

            if masked_count == 0:
                continue

            total_loss += (
                outputs.loss.item()
                * masked_count
            )

            total_loss_tokens += (
                masked_count
            )

            correct = (
                predictions[masked]
                == labels[masked]
            )

            total_correct += (
                correct.sum().item()
            )

            total_count += (
                masked_count
            )

            true_ids = (
                labels[masked]
                .detach()
                .cpu()
                .tolist()
            )

            predicted_ids = (
                predictions[masked]
                .detach()
                .cpu()
                .tolist()
            )

            for true_id, pred_id in zip(
                true_ids,
                predicted_ids
            ):

                field = (
                    id_to_field.get(
                        true_id
                    )
                )

                if field is None:
                    continue

                field_total[field] += 1

                if true_id == pred_id:
                    field_correct[field] += 1

    overall_accuracy = (
        total_correct
        / max(total_count, 1)
    )

    average_loss = (
        total_loss
        / max(
            total_loss_tokens,
            1
        )
    )

    per_field_accuracy = {}

    for field in sorted(
        field_total
    ):

        per_field_accuracy[field] = (
            field_correct[field]
            / field_total[field]
        )

    return {
        "mlm_loss": average_loss,
        "masked_accuracy":
            overall_accuracy,
        "per_field_accuracy":
            per_field_accuracy
    }

In [48]:
encoder_metrics = (
    evaluate_masked_accuracy(
        model,
        val_dataset
    )
)

encoder_metrics

{'mlm_loss': 0.6276577170494314,
 'masked_accuracy': 0.8137759799234332,
 'per_field_accuracy': {'AMOUNT': 0.30009231202507547,
  'CARD': 0.6064601839068483,
  'CHANNEL': 0.8861058748789302,
  'CITY': 0.900765991533485,
  'DELTA': 0.44347276091706483,
  'DOM': 0.9796735271765722,
  'DOW': 0.9819563894048418,
  'ERROR': 0.9839189808068612,
  'HOUR': 0.7376787890783703,
  'MCC': 0.8574382419409304,
  'MERCHANT': 0.7184516904087486,
  'MONTH': 0.9920044171037758,
  'STATE': 0.9878429593948588,
  'TIMESTAMP': 0.9999165066377222,
  'ZIP': 0.8251772253007837}}

In [49]:
encoder_metrics = (
    evaluate_masked_accuracy(
        model,
        val_dataset
    )
)

encoder_metrics

{'mlm_loss': 0.6276577170494314,
 'masked_accuracy': 0.8137759799234332,
 'per_field_accuracy': {'AMOUNT': 0.30009231202507547,
  'CARD': 0.6064601839068483,
  'CHANNEL': 0.8861058748789302,
  'CITY': 0.900765991533485,
  'DELTA': 0.44347276091706483,
  'DOM': 0.9796735271765722,
  'DOW': 0.9819563894048418,
  'ERROR': 0.9839189808068612,
  'HOUR': 0.7376787890783703,
  'MCC': 0.8574382419409304,
  'MERCHANT': 0.7184516904087486,
  'MONTH': 0.9920044171037758,
  'STATE': 0.9878429593948588,
  'TIMESTAMP': 0.9999165066377222,
  'ZIP': 0.8251772253007837}}

In [54]:
import os
import json

!mkdir "/content/financial_bert_final"

PRETRAINED_DIR = "/content/financial_bert_final"

with open(os.path.join(PRETRAINED_DIR,"vocab.json"),"w") as f:
    json.dump(token_to_id, f)

with open(os.path.join(PRETRAINED_DIR,"preprocessing.json"),"w") as f:
    json.dump(
        {
            "amount_boundaries":
                amount_boundaries.tolist(),

            "delta_boundaries":
                delta_boundaries.tolist(),

            "transactions_per_sequence":
                16,

            "tokens_per_transaction":
                TOKENS_PER_TRANSACTION,

            "max_length":
                MAX_LENGTH
        },
        f
    )

mkdir: cannot create directory ‘/content/financial_bert_final’: File exists


In [55]:
!zip -r bert_preprocess.zip /content/financial_bert_final
!zip -r bert.zip /content/financial_bert_checkpoints

  adding: content/financial_bert_final/ (stored 0%)
  adding: content/financial_bert_final/vocab.json (deflated 67%)
  adding: content/financial_bert_final/preprocessing.json (deflated 47%)
  adding: content/financial_bert_checkpoints/ (stored 0%)
  adding: content/financial_bert_checkpoints/checkpoint-37000/ (stored 0%)
  adding: content/financial_bert_checkpoints/checkpoint-37000/scheduler.pt (deflated 61%)
  adding: content/financial_bert_checkpoints/checkpoint-37000/trainer_state.json (deflated 81%)
  adding: content/financial_bert_checkpoints/checkpoint-37000/scaler.pt (deflated 64%)
  adding: content/financial_bert_checkpoints/checkpoint-37000/model.safetensors (deflated 8%)
  adding: content/financial_bert_checkpoints/checkpoint-37000/optimizer.pt (deflated 8%)
  adding: content/financial_bert_checkpoints/checkpoint-37000/config.json (deflated 51%)
  adding: content/financial_bert_checkpoints/checkpoint-37000/rng_state.pth (deflated 26%)
  adding: content/financial_bert_checkpoi

In [56]:
!pip install huggingface_hub

In [60]:
!hf auth login

? How would you like to log in?  [Use arrows, Enter to confirm]
> Log in with your browser
  Paste an access token
? How would you like to log in? Log in with your browser

    Open this URL in your browser:
        https://hf.co/oauth/device

    And enter the code: B39V-8DOK

    Waiting for authorization.......
Token is valid.
The token `oauth-kunley2` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `oauth-kunley2`
Note: This token will be refreshed automatically when it expires.


In [61]:
repo_name = "kunley2/FinBERT"
!huggingface-cli login
model.push_to_hub(repo_name)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...cwzjw9x/model.safetensors:   4%|3         |  561kB / 14.9MB            

CommitInfo(commit_url='https://huggingface.co/kunley2/FinBERT/commit/06ca467eefc6d90db093de3f01cbd0a841916df3', commit_message='Upload BertForMaskedLM', commit_description='', oid='06ca467eefc6d90db093de3f01cbd0a841916df3', pr_url=None, repo_url=RepoUrl('https://huggingface.co/kunley2/FinBERT', endpoint='https://huggingface.co', repo_type='model', repo_id='kunley2/FinBERT'), pr_revision=None, pr_num=None)